In [1]:
import numpy as np
import sys
import os
from autokmc.structure import build_surface, build_nanoparticle
from ase.visualize.x3d import view_x3d
from autokmc.surface import find_surface_atoms
import copy
from collections import Counter
from autokmc.graph import build_graph
from autokmc.site import find_adsorption_sites

In [2]:
# Ensure the project root is on the path when running from the autokmc/ subdirectory
sys.path.insert(0, os.path.abspath("../.."))

In [3]:
### Load the Allegro/NequIP calculator
from nequip.ase import NequIPCalculator

_MODEL_PATH = os.path.join(os.path.dirname(__file__) if "__file__" in dir() else ".", "asehcocuau.nequip.pt2")

def make_calc():
    """Return a fresh NequIPCalculator instance loaded from the .pt2 model."""
    return NequIPCalculator.from_compiled_model(
        compile_path=_MODEL_PATH,
        device="cuda",
        #species_to_type_name={"H": "H", "C": "C", "O": "O", "Cu": "Cu", "Au": "Au"},
    )

calc = make_calc()
print(f"Calculator : {calc.__class__.__name__}")
print(f"Model      : {_MODEL_PATH}")

Calculator : NequIPCalculator
Model      : ./asehcocuau.nequip.pt2


/g/g91/bunting4/workspace/tuolomne-venv-3.12/lib/python3.12/site-packages/nequip/ase/nequip_calculator.py:48: UserWarning: Trying to use model type names as chemical symbols; this may not be correct for your model (and may cause an error if model type names are not chemical symbols)! To avoid this warning, please provide `chemical_symbols` explicitly.
  warnings.warn(


In [4]:
## Build a Cu(111) surface slab
slab = build_surface(
    composition="Cu",
    crystal_structure="fcc",
    miller_index=(1, 1, 1),
    calculator=make_calc(),
    min_slab_size=8.0,
    min_vacuum_size=12.0,
    goal_x=12.0,
    goal_y=12.0,
    n_freeze_layers=2,
    verbose=True,
    orthogonalise=True
)

print(f"\nSlab formula : {slab.get_chemical_formula()}")
print(f"Slab atoms   : {len(slab)}")
cell = slab.get_cell()
print(f"Cell (Å)     : a={cell[0,0]:.3f}  b={cell[1,1]:.3f}  c={cell[2,2]:.3f}")

  Bulk Cu (FCC) optimised
  Converged : True  |  steps : 2
  E/atom    : -10.72548 eV
  Opt a     : 3.5922 Å

Building Cu(111) slab [FCC]  (pymatgen SlabGenerator) ...
  Orthogonal transform : (1,-1,1,1)  det=2
  Cell after ortho     : a=5.080  b=8.799  c=20.739 Å
  Tiling 3×2 → 192 atoms
  Surface Cu(111)  [FCC]
  Formula        : Cu192
  Atoms          : 192
  Cu             : 192     (100.0 %)
  Cell (Å)       : a=15.240  b=17.598  c=20.739
  z range        : [7.26, 13.48] Å
  Fixing bottom 2 layer(s): 96 atoms
  Optimisation : converged=True steps=4  E=-2017.1824 eV  E/atom=-10.5062 eV/atom

Slab formula : Cu192
Slab atoms   : 192
Cell (Å)     : a=15.240  b=17.598  c=20.739


In [5]:
## Visualise the slab
view_x3d(slab)

In [6]:
### Get surface atoms

surface_mask, surface_indices, method = find_surface_atoms(
    slab,
    which="top",
    tag_atoms=True,   # writes slab.arrays["surface"] for extxyz export
)

print(f"Detection method : {method}")
print(f"Surface atoms    : {surface_mask.sum()} / {len(slab)}")
print(f"Surface indices  : {surface_indices}")

Detection method : raycasting
Surface atoms    : 48 / 192
Surface indices  : [ 12  13  14  15  28  29  30  31  44  45  46  47  60  61  62  63  76  77
  78  79  92  93  94  95 108 109 110 111 124 125 126 127 140 141 142 143
 156 157 158 159 172 173 174 175 188 189 190 191]


In [7]:
### Visualise surface atoms in X3D
# Surface atoms are shown as Au (gold), bulk atoms remain as Cu
# so the two populations are visually distinct in x3d.
slab_vis = copy.deepcopy(slab)
symbols = np.array(slab_vis.get_chemical_symbols())
symbols[surface_indices] = "Au"
slab_vis.set_chemical_symbols(symbols.tolist())

view_x3d(slab_vis)

W0422 09:23:16.931000 3732311 /usr/WS1/bunting4/tuolomne-venv-3.12/lib/python3.12/site-packages/torch/_inductor/package/package.py:279] AOTICompiledModel deepcopy warning: AOTICompiledModel.loader is not deepcopied.


In [8]:
### Build the graph for the slab
graph = build_graph(slab)
print(f"Graph has {graph.number_of_nodes()} nodes and {graph.number_of_edges()} edges.")
# Show breakdown by node type
type_counts = Counter(d["type"] for _, d in graph.nodes(data=True))
for t, n in sorted(type_counts.items()):
    print(f"  {t:10s} : {n}")

Graph has 192 nodes and 1008 edges.
  bulk       : 144
  surface    : 48


In [9]:
### Build reactants
from autokmc.reactants import build_reactant

co   = build_reactant("[C-]#[O+]", calculator=make_calc())
o2   = build_reactant("O=O",       calculator=make_calc())
ch4  = build_reactant("C",         calculator=make_calc())
ch3  = build_reactant("[CH3]",     calculator=make_calc())
ch2  = build_reactant("[CH2]",     calculator=make_calc())
ch   = build_reactant("[CH]",      calculator=make_calc())
o    = build_reactant("[O]",       calculator=make_calc())

for r in [o, co, o2, ch4, ch3, ch2, ch]:
    print(f"\nReactant : {r.smiles}")
    print(f"  Formula  : {r.atoms.get_chemical_formula()}")
    print(f"  Atoms    : {len(r.atoms)}")
    print(f"  Nodes    : {r.graph.number_of_nodes()}")
    print(f"  Edges    : {r.graph.number_of_edges()}")
    for i, d in r.graph.nodes(data=True):
        pos = d['position']
        print(f"    node {i}  element={d['element']:2s}  type={d['type']}  r_cov={d['covalent_radius']:.3f} Å  pos=({pos[0]:.3f}, {pos[1]:.3f}, {pos[2]:.3f}) Å")


Reactant : [O]
  Formula  : O
  Atoms    : 1
  Nodes    : 1
  Edges    : 0
    node 0  element=O   type=adsorbate  r_cov=0.660 Å  pos=(6.000, 6.000, 6.000) Å

Reactant : [C-]#[O+]
  Formula  : CO
  Atoms    : 2
  Nodes    : 2
  Edges    : 1
    node 0  element=C   type=adsorbate  r_cov=0.760 Å  pos=(7.125, 6.000, 6.000) Å
    node 1  element=O   type=adsorbate  r_cov=0.660 Å  pos=(5.993, 6.000, 6.000) Å

Reactant : O=O
  Formula  : O2
  Atoms    : 2
  Nodes    : 2
  Edges    : 1
    node 0  element=O   type=adsorbate  r_cov=0.660 Å  pos=(7.190, 6.000, 6.000) Å
    node 1  element=O   type=adsorbate  r_cov=0.660 Å  pos=(5.951, 6.000, 6.000) Å

Reactant : C
  Formula  : CH4
  Atoms    : 5
  Nodes    : 5
  Edges    : 4
    node 0  element=C   type=adsorbate  r_cov=0.760 Å  pos=(6.675, 6.836, 6.581) Å
    node 1  element=H   type=adsorbate  r_cov=0.310 Å  pos=(6.002, 7.688, 6.496) Å
    node 2  element=H   type=adsorbate  r_cov=0.310 Å  pos=(6.281, 6.002, 6.002) Å
    node 3  element=H   

In [10]:
### Site visualisation helper
# Each iso-class gets a distinct proxy element so they appear in different colours.
_ISO_PROXY = ["Au", "Pt", "Pd", "Ag", "Ir", "Rh", "Os", "Re"]

def _make_site_vis(slab, sites, reactant):
    """Return an Atoms object: slab + one representative per iso-class."""
    import copy as _copy
    vis = _copy.deepcopy(slab)
    seen_classes = set()
    for s in sites:
        if s.iso_class in seen_classes:
            continue
        seen_classes.add(s.iso_class)
        proxy = _ISO_PROXY[s.iso_class % len(_ISO_PROXY)]
        if len(reactant.atoms) == 1:
            vis.append(proxy)
            vis.positions[-1] = s.position
        else:
            ads_syms = reactant.atoms.get_chemical_symbols()
            for k, sym in enumerate(ads_syms):
                vis.append(proxy if k == 0 else sym)
                vis.positions[-1] = s.position[k]
    return vis

def _print_sites_single(label, sites):
    print(f"\n{label} sites summary")
    print(f"  Total sites  : {len(sites)}")
    for iso_cid, count in sorted(Counter(s.iso_class for s in sites).items()):
        rep = next(s for s in sites if s.iso_class == iso_cid)
        e_str = f"  E_ads={rep.energy:.4f} eV  converged={rep.converged}" if rep.energy is not None else ""
        print(f"  iso-class {iso_cid:2d}  {rep.site_type:8s}  {count:3d} sites{e_str}  "
              f"pos = ({rep.position[0]:.3f}, {rep.position[1]:.3f}, {rep.position[2]:.3f}) Å")

def _print_sites_multi(label, sites):
    print(f"\n{label} sites summary")
    print(f"  Total sites  : {len(sites)}")
    for iso_cid, count in sorted(Counter(s.iso_class for s in sites).items()):
        rep = next(s for s in sites if s.iso_class == iso_cid)
        n_surf = len({sg for _, sg in rep.conn_global})
        e_str = f"  E_ads={rep.energy:.4f} eV  converged={rep.converged}" if rep.energy is not None else ""
        print(f"  iso-class {iso_cid:2d}  {rep.site_type:8s}  {count:3d} sites  "
              f"{n_surf} surface atoms bonded{e_str}")

In [11]:
### Find adsorption sites — O (single atom, k_max=4)

sites_o, site_graph_o = find_adsorption_sites(
    graph, o,
    k_max=4,
    calculator=make_calc(),
    slab=slab,
    verbose=True,
)
_print_sites_single("O", sites_o)

Finding sites for '[O]'  (single-atom path,  k_max=4)
  [single-graph] co-bond graph: 48 nodes, 144 edges  (r_cov_ads=0.660 Å, bond_factor=1.1)
  [single-graph] clique candidates: 288  (k=1: 48  k=2: 144  k=3: 96)
  [single-graph] iso-classes: 4


W0422 09:23:18.348000 3732311 /usr/WS1/bunting4/tuolomne-venv-3.12/lib/python3.12/site-packages/torch/_inductor/package/package.py:279] AOTICompiledModel deepcopy warning: AOTICompiledModel.loader is not deepcopied.
W0422 09:23:18.377000 3732311 /usr/WS1/bunting4/tuolomne-venv-3.12/lib/python3.12/site-packages/torch/_inductor/package/package.py:279] AOTICompiledModel deepcopy warning: AOTICompiledModel.loader is not deepcopied.
W0422 09:23:18.433000 3732311 /usr/WS1/bunting4/tuolomne-venv-3.12/lib/python3.12/site-packages/torch/_inductor/package/package.py:279] AOTICompiledModel deepcopy warning: AOTICompiledModel.loader is not deepcopied.



  EMT relaxation: 1 representative per iso-class (4 classes, 96 atoms frozen)
  iso-class  0  [OK  ]  converged=True  intended=1-fold  actual=1-fold  E_ads=-5.8610 eV


W0422 09:23:18.555000 3732311 /usr/WS1/bunting4/tuolomne-venv-3.12/lib/python3.12/site-packages/torch/_inductor/package/package.py:279] AOTICompiledModel deepcopy warning: AOTICompiledModel.loader is not deepcopied.


  iso-class  1  [FAIL]  converged=True  intended=2-fold  actual=3-fold


W0422 09:23:18.595000 3732311 /usr/WS1/bunting4/tuolomne-venv-3.12/lib/python3.12/site-packages/torch/_inductor/package/package.py:279] AOTICompiledModel deepcopy warning: AOTICompiledModel.loader is not deepcopied.


  iso-class  2  [OK  ]  converged=True  intended=3-fold  actual=3-fold  E_ads=-8.0091 eV
  iso-class  3  [OK  ]  converged=True  intended=3-fold  actual=3-fold  E_ads=-7.9422 eV
  Valid classes : 3 / 4

Site classification
  iso-class  0  top        48 sites  E_ads=-5.8610 eV  converged=True
  iso-class  2  hollow     48 sites  E_ads=-8.0091 eV  converged=True
  iso-class  3  hollow     48 sites  E_ads=-7.9422 eV  converged=True
  Total unique sites : 144
  Adjacency edges    : 864

O sites summary
  Total sites  : 144
  iso-class  0  top        48 sites  E_ads=-5.8610 eV  converged=True  pos = (0.000, 7.332, 15.619) Å
  iso-class  2  hollow     48 sites  E_ads=-8.0091 eV  converged=True  pos = (1.270, 6.599, 14.771) Å
  iso-class  3  hollow     48 sites  E_ads=-7.9422 eV  converged=True  pos = (1.270, 8.066, 14.771) Å


In [12]:
slab_o_vis = _make_site_vis(slab, sites_o, o)
print(f"O site visualisation: {len(slab_o_vis)} atoms "
      f"({len(slab)} slab + {len(slab_o_vis) - len(slab)} representatives)")
for iso_cid in sorted({s.iso_class for s in sites_o}):
    rep = next(s for s in sites_o if s.iso_class == iso_cid)
    proxy = _ISO_PROXY[iso_cid % len(_ISO_PROXY)]
    e_str = f"  E_ads={rep.energy:.4f} eV" if rep.energy is not None else ""
    print(f"  iso-class {iso_cid}  ({rep.site_type})  proxy={proxy}{e_str}")
view_x3d(slab_o_vis)

W0422 09:23:18.658000 3732311 /usr/WS1/bunting4/tuolomne-venv-3.12/lib/python3.12/site-packages/torch/_inductor/package/package.py:279] AOTICompiledModel deepcopy warning: AOTICompiledModel.loader is not deepcopied.


O site visualisation: 195 atoms (192 slab + 3 representatives)
  iso-class 0  (top)  proxy=Au  E_ads=-5.8610 eV
  iso-class 2  (hollow)  proxy=Pd  E_ads=-8.0091 eV
  iso-class 3  (hollow)  proxy=Ag  E_ads=-7.9422 eV


In [13]:
### Find adsorption sites — CO (multi-atom, k_max=3)

sites_co, site_graph_co = find_adsorption_sites(
    graph, co,
    k_max=3,
    calculator=make_calc(),
    slab=slab,
    verbose=True,
)
_print_sites_multi("CO", sites_co)

Finding sites for '[C-]#[O+]'  (multi-atom path,  k_max=3)
  [multi-graph] unique connectivity patterns: 12494
  [multi-graph] iso-classes: 429


W0422 09:23:50.207000 3732311 /usr/WS1/bunting4/tuolomne-venv-3.12/lib/python3.12/site-packages/torch/_inductor/package/package.py:279] AOTICompiledModel deepcopy warning: AOTICompiledModel.loader is not deepcopied.
W0422 09:23:50.243000 3732311 /usr/WS1/bunting4/tuolomne-venv-3.12/lib/python3.12/site-packages/torch/_inductor/package/package.py:279] AOTICompiledModel deepcopy warning: AOTICompiledModel.loader is not deepcopied.
/g/g91/bunting4/workspace/tuolomne-venv-3.12/lib/python3.12/site-packages/ase/io/extxyz.py:318: UserWarning: Skipping unhashable information frozen_indices
  warnings.warn('Skipping unhashable information '



  Calculator relaxation: 1 representative per iso-class (429 classes, 2-atom adsorbate, 96 atoms frozen)


W0422 09:23:50.475000 3732311 /usr/WS1/bunting4/tuolomne-venv-3.12/lib/python3.12/site-packages/torch/_inductor/package/package.py:279] AOTICompiledModel deepcopy warning: AOTICompiledModel.loader is not deepcopied.
W0422 09:23:50.619000 3732311 /usr/WS1/bunting4/tuolomne-venv-3.12/lib/python3.12/site-packages/torch/_inductor/package/package.py:279] AOTICompiledModel deepcopy warning: AOTICompiledModel.loader is not deepcopied.


  iso-class  0  [OK  ]  converged=True  surf-bonds intended=1  actual=1  E_ads=-17.1721 eV
  iso-class  1  [FAIL]  converged=True  surf-bonds intended=1  actual=1


...

  iso-class 425  [FAIL]  converged=True  surf-bonds intended=4  actual=5


W0422 09:26:49.256000 3732311 /usr/WS1/bunting4/tuolomne-venv-3.12/lib/python3.12/site-packages/torch/_inductor/package/package.py:279] AOTICompiledModel deepcopy warning: AOTICompiledModel.loader is not deepcopied.


  iso-class 426  [FAIL]  converged=True  surf-bonds intended=5  actual=5


W0422 09:26:49.583000 3732311 /usr/WS1/bunting4/tuolomne-venv-3.12/lib/python3.12/site-packages/torch/_inductor/package/package.py:279] AOTICompiledModel deepcopy warning: AOTICompiledModel.loader is not deepcopied.


  iso-class 427  [FAIL]  converged=True  surf-bonds intended=5  actual=5
  iso-class 428  [FAIL]  converged=True  surf-bonds intended=5  actual=3
  Valid classes : 4 / 429

Site classification
  iso-class  0  top        48 sites  E_ads=-17.1721 eV  converged=True
  iso-class 17  bridge    144 sites  E_ads=-17.3179 eV  converged=True
  iso-class 122  hollow     48 sites  E_ads=-17.4429 eV  converged=True
  iso-class 126  hollow     48 sites  E_ads=-17.3988 eV  converged=True
  Total unique sites : 288
  Adjacency edges    : 3312

CO sites summary
  Total sites  : 288
  iso-class  0  top        48 sites  1 surface atoms bonded  E_ads=-17.1721 eV  converged=True
  iso-class 17  bridge    144 sites  2 surface atoms bonded  E_ads=-17.3179 eV  converged=True
  iso-class 122  hollow     48 sites  3 surface atoms bonded  E_ads=-17.4429 eV  converged=True
  iso-class 126  hollow     48 sites  3 surface atoms bonded  E_ads=-17.3988 eV  converged=True


In [14]:
slab_co_vis = _make_site_vis(slab, sites_co, co)
print(f"CO site visualisation: {len(slab_co_vis)} atoms "
      f"({len(slab)} slab + {len(slab_co_vis) - len(slab)} representatives)")
for iso_cid in sorted({s.iso_class for s in sites_co}):
    rep = next(s for s in sites_co if s.iso_class == iso_cid)
    proxy = _ISO_PROXY[iso_cid % len(_ISO_PROXY)]
    e_str = f"  E_ads={rep.energy:.4f} eV" if rep.energy is not None else ""
    print(f"  iso-class {iso_cid}  ({rep.site_type})  proxy={proxy}{e_str}")
view_x3d(slab_co_vis)

W0422 09:26:50.068000 3732311 /usr/WS1/bunting4/tuolomne-venv-3.12/lib/python3.12/site-packages/torch/_inductor/package/package.py:279] AOTICompiledModel deepcopy warning: AOTICompiledModel.loader is not deepcopied.


CO site visualisation: 200 atoms (192 slab + 8 representatives)
  iso-class 0  (top)  proxy=Au  E_ads=-17.1721 eV
  iso-class 17  (bridge)  proxy=Pt  E_ads=-17.3179 eV
  iso-class 122  (hollow)  proxy=Pd  E_ads=-17.4429 eV
  iso-class 126  (hollow)  proxy=Os  E_ads=-17.3988 eV


In [15]:
### Find adsorption sites — O2 (multi-atom, k_max=4)

sites_o2, site_graph_o2 = find_adsorption_sites(
    graph, o2,
    k_max=4,
    calculator=make_calc(),
    slab=slab,
    verbose=True,
)
_print_sites_multi("O2", sites_o2)

Finding sites for 'O=O'  (multi-atom path,  k_max=4)
  [multi-graph] unique connectivity patterns: 4769
  [multi-graph] iso-classes: 84


W0422 09:26:57.980000 3732311 /usr/WS1/bunting4/tuolomne-venv-3.12/lib/python3.12/site-packages/torch/_inductor/package/package.py:279] AOTICompiledModel deepcopy warning: AOTICompiledModel.loader is not deepcopied.
W0422 09:26:58.012000 3732311 /usr/WS1/bunting4/tuolomne-venv-3.12/lib/python3.12/site-packages/torch/_inductor/package/package.py:279] AOTICompiledModel deepcopy warning: AOTICompiledModel.loader is not deepcopied.



  Calculator relaxation: 1 representative per iso-class (84 classes, 2-atom adsorbate, 96 atoms frozen)


W0422 09:26:58.395000 3732311 /usr/WS1/bunting4/tuolomne-venv-3.12/lib/python3.12/site-packages/torch/_inductor/package/package.py:279] AOTICompiledModel deepcopy warning: AOTICompiledModel.loader is not deepcopied.
W0422 09:26:58.543000 3732311 /usr/WS1/bunting4/tuolomne-venv-3.12/lib/python3.12/site-packages/torch/_inductor/package/package.py:279] AOTICompiledModel deepcopy warning: AOTICompiledModel.loader is not deepcopied.


  iso-class  0  [FAIL]  converged=True  surf-bonds intended=1  actual=3
  iso-class  1  [FAIL]  converged=True  surf-bonds intended=1  actual=1


...

  iso-class 80  [FAIL]  converged=True  surf-bonds intended=4  actual=4


W0422 09:27:42.652000 3732311 /usr/WS1/bunting4/tuolomne-venv-3.12/lib/python3.12/site-packages/torch/_inductor/package/package.py:279] AOTICompiledModel deepcopy warning: AOTICompiledModel.loader is not deepcopied.


  iso-class 81  [FAIL]  converged=True  surf-bonds intended=5  actual=4


W0422 09:27:42.864000 3732311 /usr/WS1/bunting4/tuolomne-venv-3.12/lib/python3.12/site-packages/torch/_inductor/package/package.py:279] AOTICompiledModel deepcopy warning: AOTICompiledModel.loader is not deepcopied.


  iso-class 82  [FAIL]  converged=True  surf-bonds intended=5  actual=4
  iso-class 83  [FAIL]  converged=True  surf-bonds intended=5  actual=5
  Valid classes : 0 / 84

Site classification
  Total unique sites : 0
  Adjacency edges    : 0

O2 sites summary
  Total sites  : 0


In [16]:
slab_o2_vis = _make_site_vis(slab, sites_o2, o2)
print(f"O2 site visualisation: {len(slab_o2_vis)} atoms "
      f"({len(slab)} slab + {len(slab_o2_vis) - len(slab)} representatives)")
for iso_cid in sorted({s.iso_class for s in sites_o2}):
    rep = next(s for s in sites_o2 if s.iso_class == iso_cid)
    proxy = _ISO_PROXY[iso_cid % len(_ISO_PROXY)]
    e_str = f"  E_ads={rep.energy:.4f} eV" if rep.energy is not None else ""
    print(f"  iso-class {iso_cid}  ({rep.site_type})  proxy={proxy}{e_str}")
view_x3d(slab_o2_vis)

W0422 09:27:44.704000 3732311 /usr/WS1/bunting4/tuolomne-venv-3.12/lib/python3.12/site-packages/torch/_inductor/package/package.py:279] AOTICompiledModel deepcopy warning: AOTICompiledModel.loader is not deepcopied.


O2 site visualisation: 192 atoms (192 slab + 0 representatives)


In [ ]:
### Find adsorption sites — CH4 (multi-atom, k_max=4)

sites_ch4, site_graph_ch4 = find_adsorption_sites(
    graph, ch4,
    k_max=4,
    calculator=make_calc(),
    slab=slab,
    verbose=True,
)
_print_sites_multi("CH4", sites_ch4)

Finding sites for 'C'  (multi-atom path,  k_max=4)


In [ ]:
slab_ch4_vis = _make_site_vis(slab, sites_ch4, ch4)
print(f"CH4 site visualisation: {len(slab_ch4_vis)} atoms "
      f"({len(slab)} slab + {len(slab_ch4_vis) - len(slab)} representatives)")
for iso_cid in sorted({s.iso_class for s in sites_ch4}):
    rep = next(s for s in sites_ch4 if s.iso_class == iso_cid)
    proxy = _ISO_PROXY[iso_cid % len(_ISO_PROXY)]
    e_str = f"  E_ads={rep.energy:.4f} eV" if rep.energy is not None else ""
    print(f"  iso-class {iso_cid}  ({rep.site_type})  proxy={proxy}{e_str}")
view_x3d(slab_ch4_vis)

In [ ]:
### Find adsorption sites — CH3 (multi-atom, k_max=4)

sites_ch3, site_graph_ch3 = find_adsorption_sites(
    graph, ch3,
    k_max=4,
    calculator=make_calc(),
    slab=slab,
    verbose=True,
)
_print_sites_multi("CH3", sites_ch3)

In [ ]:
slab_ch3_vis = _make_site_vis(slab, sites_ch3, ch3)
print(f"CH3 site visualisation: {len(slab_ch3_vis)} atoms "
      f"({len(slab)} slab + {len(slab_ch3_vis) - len(slab)} representatives)")
for iso_cid in sorted({s.iso_class for s in sites_ch3}):
    rep = next(s for s in sites_ch3 if s.iso_class == iso_cid)
    proxy = _ISO_PROXY[iso_cid % len(_ISO_PROXY)]
    e_str = f"  E_ads={rep.energy:.4f} eV" if rep.energy is not None else ""
    print(f"  iso-class {iso_cid}  ({rep.site_type})  proxy={proxy}{e_str}")
view_x3d(slab_ch3_vis)

In [ ]:
### Find adsorption sites — CH2 (multi-atom, k_max=4)

sites_ch2, site_graph_ch2 = find_adsorption_sites(
    graph, ch2,
    k_max=4,
    calculator=make_calc(),
    slab=slab,
    verbose=True,
)
_print_sites_multi("CH2", sites_ch2)

In [ ]:
slab_ch2_vis = _make_site_vis(slab, sites_ch2, ch2)
print(f"CH2 site visualisation: {len(slab_ch2_vis)} atoms "
      f"({len(slab)} slab + {len(slab_ch2_vis) - len(slab)} representatives)")
for iso_cid in sorted({s.iso_class for s in sites_ch2}):
    rep = next(s for s in sites_ch2 if s.iso_class == iso_cid)
    proxy = _ISO_PROXY[iso_cid % len(_ISO_PROXY)]
    e_str = f"  E_ads={rep.energy:.4f} eV" if rep.energy is not None else ""
    print(f"  iso-class {iso_cid}  ({rep.site_type})  proxy={proxy}{e_str}")
view_x3d(slab_ch2_vis)

In [ ]:
### Find adsorption sites — CH (multi-atom, k_max=4)

sites_ch, site_graph_ch = find_adsorption_sites(
    graph, ch,
    k_max=4,
    calculator=make_calc(),
    slab=slab,
    verbose=True,
)
_print_sites_multi("CH", sites_ch)

In [ ]:
slab_ch_vis = _make_site_vis(slab, sites_ch, ch)
print(f"CH site visualisation: {len(slab_ch_vis)} atoms "
      f"({len(slab)} slab + {len(slab_ch_vis) - len(slab)} representatives)")
for iso_cid in sorted({s.iso_class for s in sites_ch}):
    rep = next(s for s in sites_ch if s.iso_class == iso_cid)
    proxy = _ISO_PROXY[iso_cid % len(_ISO_PROXY)]
    e_str = f"  E_ads={rep.energy:.4f} eV" if rep.energy is not None else ""
    print(f"  iso-class {iso_cid}  ({rep.site_type})  proxy={proxy}{e_str}")
view_x3d(slab_ch_vis)